In [2]:
import json
import os
import pandas as pd

In [3]:
PATH_CLEANED = "listings_cleaned_keer.csv"
OUTPUT_DIR   = "."
CUTOFF_DATE  = "2024-10-18"

# 4 LLM output JSONs (comment out any not ready yet)
LLM_JOBS = [
    {"label": "qwen_desc",      "json_path": "inference_Qwen3_DESCRIPTION_ONLY.json"},
    {"label": "qwen_combined",  "json_path": "inference_Qwen3_COMBINED.json"},
    {"label": "nemo_desc",      "json_path": "inference_NEMO_DESCRIPTION_ONLY.json"},
    {"label": "nemo_combined",  "json_path": "inference_NEMO_COMBINED.json"},
]

In [4]:
def unique_lowercase(lst):
    """Deduplicate by lowercase+strip, preserve original casing and order."""
    if not isinstance(lst, list):
        return []
    seen = set()
    uniq = []
    for s in lst:
        key = str(s).lower().strip()
        if key and key not in seen:
            seen.add(key)
            uniq.append(str(s).strip())
    return uniq

def list_to_str(lst):
    return "; ".join(lst) if isinstance(lst, list) else ""

def combined_unique_count(sp_list, gen_list):
    """Count unique locations across specific + general (matches ML logic)."""
    combined = [s.lower().strip() for s in sp_list] + \
               [s.lower().strip() for s in gen_list]
    return len({x for x in combined if x})

In [5]:
print("── Loading cleaned 2024-10 pool ──")
cleaned = pd.read_csv(PATH_CLEANED, usecols=["id"])
cleaned["id"] = cleaned["id"].astype(str)
print(f"  Listings in cleaned: {len(cleaned)}\n")

── Loading cleaned 2024-10 pool ──
  Listings in cleaned: 5138



In [6]:
def process_one_job(job):
    print(f"── {job['label']} ──")

    if not os.path.exists(job["json_path"]):
        print(f"  File not found: {job['json_path']} — skipping\n")
        return

    # Load JSON
    with open(job["json_path"], "r") as f:
        llm_raw = json.load(f)
    print(f"  Total JSON records: {len(llm_raw)}")

    # Flatten to DataFrame with per-category deduplicated lists
    llm_df = pd.DataFrame([{
        "id":      str(r["id"]),
        "scraped": r.get("scraped", ""),
        "desc":    r.get("desc", ""),
        "specific_list": unique_lowercase(r.get("specific_locations", [])),
        "general_list":  unique_lowercase(r.get("general_references", [])),
        "parent_list":   unique_lowercase(r.get("parent_references",  [])),
    } for r in llm_raw])

    # Filter to pre-cutoff records
    llm_df_valid = llm_df[llm_df["scraped"] <= CUTOFF_DATE].copy()
    print(f"  Records within cutoff: {len(llm_df_valid)}")

    # Keep latest per listing
    llm_latest = (
        llm_df_valid.sort_values("scraped")
                    .groupby("id", as_index=False)
                    .last()
    )
    print(f"  Unique listings after dedup: {len(llm_latest)}")

    # Inner join to 2024-10 pool
    matched = cleaned.merge(llm_latest, on="id", how="inner")
    print(f"  Matched to cleaned: {len(matched)} / {len(cleaned)} "
          f"({len(matched)/len(cleaned)*100:.1f}%)")

    # Counts per category
    matched["llm_n_specific"] = matched["specific_list"].apply(len)
    matched["llm_n_general"]  = matched["general_list"].apply(len)
    matched["llm_n_parent"]   = matched["parent_list"].apply(len)

    # llm_score = combined unique (specific + general), matches ML spatial_semantics
    matched["llm_score"] = matched.apply(
        lambda r: combined_unique_count(r["specific_list"], r["general_list"]),
        axis=1
    )

    # String versions for CSV
    matched["llm_specific"] = matched["specific_list"].apply(list_to_str)
    matched["llm_general"]  = matched["general_list"].apply(list_to_str)
    matched["llm_parent"]   = matched["parent_list"].apply(list_to_str)

    # Build output
    out_df = pd.DataFrame({
        "id":             matched["id"],
        "scraped":        matched["scraped"],
        "description":    matched["desc"],
        "llm_specific":   matched["llm_specific"],
        "llm_general":    matched["llm_general"],
        "llm_parent":     matched["llm_parent"],
        "llm_n_specific": matched["llm_n_specific"],
        "llm_n_general":  matched["llm_n_general"],
        "llm_n_parent":   matched["llm_n_parent"],
        "llm_score":      matched["llm_score"],
    })

    out_path = os.path.join(OUTPUT_DIR, f"llm_score_{job['label']}_202410.csv")
    out_df.to_csv(out_path, index=False)

    stats = out_df["llm_score"].describe()
    print(f"  llm_score — mean: {stats['mean']:.2f}  median: {stats['50%']:.0f}  "
          f"max: {stats['max']:.0f}  zero: {(out_df['llm_score']==0).sum()}")
    print(f"  Saved: {out_path}\n")

In [7]:
for job in LLM_JOBS:
    process_one_job(job)

print("Done.")


── qwen_desc ──
  Total JSON records: 37058
  Records within cutoff: 32009
  Unique listings after dedup: 15011
  Matched to cleaned: 4189 / 5138 (81.5%)
  llm_score — mean: 5.34  median: 5  max: 26  zero: 327
  Saved: ./llm_score_qwen_desc_202410.csv

── qwen_combined ──
  Total JSON records: 38024
  Records within cutoff: 32895
  Unique listings after dedup: 15075
  Matched to cleaned: 4211 / 5138 (82.0%)
  llm_score — mean: 6.84  median: 6  max: 26  zero: 255
  Saved: ./llm_score_qwen_combined_202410.csv

── nemo_desc ──
  Total JSON records: 37058
  Records within cutoff: 32009
  Unique listings after dedup: 15011
  Matched to cleaned: 4189 / 5138 (81.5%)
  llm_score — mean: 1.74  median: 1  max: 17  zero: 1810
  Saved: ./llm_score_nemo_desc_202410.csv

── nemo_combined ──
  Total JSON records: 38024
  Records within cutoff: 32895
  Unique listings after dedup: 15075
  Matched to cleaned: 4211 / 5138 (82.0%)
  llm_score — mean: 2.17  median: 2  max: 17  zero: 1646
  Saved: ./llm_sc